[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_23_linear_pure.ipynb)

# 🟢 Easy: Linear Layer without Flax

*Core Ops & Layers*
Problem 03's linear layer, written with no Flax.

### Signature
```python
class SimpleLinear:
    def __init__(self, in_features, out_features, *, key, use_bias=True): ...
    def __call__(self, x): ...        # (..., in_features) -> (..., out_features)
```

Same class name, same argument names, same attributes as the `nnx` version:

| | |
|---|---|
| `self.kernel` | `(in_features, out_features)`, scaled by `1/sqrt(in_features)` |
| `self.bias` | `(out_features,)` zeros, or **`None`** when `use_bias=False` |

`kernel` is `(in, out)` — the Flax layout, the transpose of PyTorch's — which
is why `__call__` is `x @ self.kernel` with no transpose anywhere.

### Any leading shape
`(in,)`, `(N, in)` and `(B, T, in)` all work for free, as long as `__call__`
never mentions the batch axes.

### Why this exists alongside problem 03
Interview sandboxes (CoderPad and friends) often ship `jax` and nothing else,
which makes every `nnx.Module` problem here unrunnable there.

**The API is deliberately as close to `nnx` as it can be** — same class name,
same argument names, same attribute names, same array layouts. Only the source
of randomness changes:

```python
nnx.Linear(4, 3, rngs=nnx.Rngs(params=0))    # nnx
SimpleLinear(4, 3, key=jax.random.key(0))    # here
```

so practising this reinforces the `nnx` version instead of competing with it.

### What you give up
A plain Python class is **not a pytree**, so `jax.grad(loss)(layer)` does not
work. Differentiate with respect to the input, or keep the arrays outside the
object.

Rebinding `self.kernel` to a tracer inside a traced function looks like a way
around that — it even returns the right gradient once — and then leaks the
tracer into the next call:

```
UnexpectedTracerError: A function transformed by JAX had a side effect ...
```

That leak is exactly the problem Flax and Equinox exist to solve. In an
interview you are almost always asked for the forward pass, so this trade is
usually free.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


class SimpleLinear:
    """y = x @ kernel + bias, with the arrays built by hand."""

    def __init__(self, in_features, out_features, *, key, use_bias=True):
        pass  # Replace this

    def __call__(self, x):
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

layer = SimpleLinear(4, 3, key=jax.random.key(0))
print("kernel", layer.kernel.shape, " bias", layer.bias.shape)

for shape in [(4,), (10, 4), (2, 5, 4)]:
    print(f"  {str(shape):<10} -> {layer(jnp.ones(shape)).shape}")

no_bias = SimpleLinear(4, 3, key=jax.random.key(0), use_bias=False)
print("\nuse_bias=False -> bias is", no_bias.bias)

# A plain class is not a pytree, so differentiate w.r.t. the INPUT.
g = jax.grad(lambda v: jnp.sum(layer(v)))(jnp.ones((2, 4)))
print("d/dx shape:", g.shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("linear_pure")

# hint("linear_pure")      # stuck? nudge without the answer
# solution("linear_pure")  # spoiler: the reference implementation
# status()                 # your dashboard across all problems